# Wikipedia airborne-radar knowledge graph with Neo4j

This notebook uses the repository's `combat_id_calibration.graph_ingest` module to build a local Neo4j knowledge graph from Wikipedia pages for the MiG-29, the Ukrainian Air Force, the Russian Air Force, the Bars radar, and representative Russian or Israeli airborne radars.

The existing ingestion module performs the core workflow: fetch Wikipedia HTML, extract readable text, chunk documents, ask a local Ollama model to return auditable JSON facts, optionally write facts to JSONL, and populate Neo4j with `Entity`, `Source`, `FACT`, and `MENTIONED_IN` records.

> Wikipedia is a convenient public source, but it is not authoritative. Review `facts-jsonl` output before using extracted relationships for combat-identification scoring or calibration.


## 1. Install and runtime prerequisites

From the repository root, install the graph extra and make sure Ollama is running with the configured model:

```bash
python -m pip install -e .[graph]
ollama pull qwen3.5:9b
ollama serve
```

If your notebook starts in `notebooks/`, the setup cell below adds the repository root to `sys.path` so it imports the local module under development.


In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.graph_ingest import (
    DEFAULT_MODEL,
    DEFAULT_OLLAMA_URL,
    extract_facts,
    load_documents,
    populate_neo4j,
    write_facts_jsonl,
)


## 2. Configure Wikipedia sources

The first four URLs satisfy the explicitly requested pages. The remaining URLs add representative Russian and Israeli airborne radar pages so the resulting graph has more radar-specific evidence.


In [2]:
NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = "password123"  # replace with your Neo4j password before running the population cell
NEO4J_DATABASE = None
FACTS_JSONL = REPO_ROOT / 'wikipedia_airborne_radars_facts.jsonl'
MODEL = DEFAULT_MODEL
OLLAMA_URL = DEFAULT_OLLAMA_URL
MAX_CHARS = 6000
OVERLAP = 500

WIKIPEDIA_URLS = [
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29',
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force',
    'https://en.wikipedia.org/wiki/Russian_Air_Force',
    'https://en.wikipedia.org/wiki/Bars_radar',

    # Additional Russian airborne radars
    'https://en.wikipedia.org/wiki/Irbis-E',
    'https://en.wikipedia.org/wiki/Zhuk_(radar)',
    'https://en.wikipedia.org/wiki/Mech_radar',
    # Additional Israeli airborne radars
    'https://en.wikipedia.org/wiki/EL/M-2032',
    'https://en.wikipedia.org/wiki/EL/W-2085',
    'https://en.wikipedia.org/wiki/EL/M-2052',
]


In [3]:
WIKIPEDIA_URLS_2 = ['https://en.wikipedia.org/wiki/Sukhoi_Su-27',
                    'https://en.wikipedia.org/wiki/McDonnell_Douglas_F-15_Eagle',
                    'https://en.wikipedia.org/wiki/Eurofighter_Typhoon',
                    'https://en.wikipedia.org/wiki/R-77',
                    'https://en.wikipedia.org/wiki/Euroradar_CAPTOR'
                    ]

## 3. Load Wikipedia documents with `graph_ingest.py`

`load_documents` delegates each Wikipedia URL to `read_wikipedia`, preserving a deterministic source ID, source type, locator URL, article title, and extracted article text.


In [4]:


documents_2 = load_documents(pdf_paths=[], wikipedia_urls=WIKIPEDIA_URLS_2)
print(f'Loaded {len(documents_2)} Wikipedia documents')
for document in documents_2:
    print(f'- {document.title}: {len(document.text):,} characters from {document.locator}')

https://en.wikipedia.org/wiki/Sukhoi_Su-27
https://en.wikipedia.org/wiki/McDonnell_Douglas_F-15_Eagle
https://en.wikipedia.org/wiki/Eurofighter_Typhoon
https://en.wikipedia.org/wiki/R-77
https://en.wikipedia.org/wiki/Euroradar_CAPTOR
Loaded 5 Wikipedia documents
- Sukhoi Su-27: 85,407 characters from https://en.wikipedia.org/wiki/Sukhoi_Su-27
- McDonnell Douglas F-15 Eagle: 108,200 characters from https://en.wikipedia.org/wiki/McDonnell_Douglas_F-15_Eagle
- Eurofighter Typhoon: 155,830 characters from https://en.wikipedia.org/wiki/Eurofighter_Typhoon
- R-77: 21,727 characters from https://en.wikipedia.org/wiki/R-77
- Euroradar CAPTOR: 41,847 characters from https://en.wikipedia.org/wiki/Euroradar_CAPTOR


## 4. Extract auditable facts with Ollama

This cell calls the repository ingestion module's `extract_facts`, which chunks each source and applies the module's constrained JSON extraction prompt. Keep `FACTS_JSONL` under review; it is the audit artifact to inspect before trusting the Neo4j graph.


In [ ]:
facts = extract_facts(
    documents_2,
    model=MODEL,
    ollama_url=OLLAMA_URL,
    max_chars=1000,
    overlap=200,
)
write_facts_jsonl(facts, FACTS_JSONL)
print(f'Extracted {len(facts)} facts')
print(f'Wrote review file: {FACTS_JSONL}')


[graph-ingest] starting fact extraction with model='qwen3.5:9b', ollama_url='http://localhost:11434', max_chars=1000, overlap=200
[graph-ingest] document 1: title='Sukhoi Su-27', source_id=0a6672b89f918010, type=wikipedia, text_chars=85407, chunks=105
[graph-ingest] document 1 chunk 1/105: sending 1000 chars to Ollama
[graph-ingest] document 1 chunk 1/105: received 12 chars; preview='{"facts":[]}'
[graph-ingest] document 1 chunk 1/105: JSON candidate 1 contains facts=0
[graph-ingest] document 1 chunk 1/105: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 1/105: added 0 fact(s)
[graph-ingest] document 1 chunk 2/105: sending 1000 chars to Ollama
[graph-ingest] document 1 chunk 2/105: received 12 chars; preview='{"facts":[]}'
[graph-ingest] document 1 chunk 2/105: JSON candidate 1 contains facts=0
[graph-ingest] document 1 chunk 2/105: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 2/105: added 0 fact(s)
[graph-ingest] docum

## 5. Populate Neo4j with `graph_ingest.py`

`populate_neo4j` creates the same schema used by the repository CLI:

- `Entity(id, name)`
- `Source(id, source_type, locator)`
- `FACT(predicate, source_id, evidence, confidence)`
- `MENTIONED_IN`

If you want a clean graph, clear the target Neo4j database before running this population cell.

If `NEO4J_PASSWORD` is left as `None` or an empty string, the population cell stops before opening a Neo4j driver and asks you to set the password.


In [8]:
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = "password123"

populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print(f'Populated Neo4j database: {NEO4J_URI}')


Populated Neo4j database: bolt://localhost:7687


## 6. Query examples

The queries below inspect the graph generated by `populate_neo4j`. They intentionally work with the generic `Entity`/`FACT` schema from `graph_ingest.py` instead of introducing a separate notebook-only schema.


In [9]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
session_kwargs = {'database': NEO4J_DATABASE} if NEO4J_DATABASE else {}
session = driver.session(**session_kwargs)

def show(query: str, params: dict | None = None):
    result = session.run(query, params or {})
    rows = pd.DataFrame([record.data() for record in result])
    display(rows)
    return rows


In [10]:
show('''
MATCH (source:Source)
RETURN source.source_type AS source_type, source.locator AS locator
ORDER BY locator
''')


,source_type,locator
0,wikipedia,https://en.wikipedia.org/wiki/Bars_radar
1,wikipedia,https://en.wikipedia.org/wiki/EL/M-2032
2,wikipedia,https://en.wikipedia.org/wiki/EL/M-2052
3,wikipedia,https://en.wikipedia.org/wiki/EL/W-2085
4,wikipedia,https://en.wikipedia.org/wiki/Irbis-E
5,wikipedia,https://en.wikipedia.org/wiki/Mech_radar
6,wikipedia,https://en.wikipedia.org/wiki/Mikoyan_MiG-29
7,wikipedia,https://en.wikipedia.org/wiki/Russian_Air_Force
8,wikipedia,https://en.wikipedia.org/wiki/Ukrainian_Air_Force
9,wikipedia,https://en.wikipedia.org/wiki/Zhuk_(radar)


,source_type,locator
0,wikipedia,https://en.wikipedia.org/wiki/Bars_radar
1,wikipedia,https://en.wikipedia.org/wiki/EL/M-2032
2,wikipedia,https://en.wikipedia.org/wiki/EL/M-2052
3,wikipedia,https://en.wikipedia.org/wiki/EL/W-2085
4,wikipedia,https://en.wikipedia.org/wiki/Irbis-E
5,wikipedia,https://en.wikipedia.org/wiki/Mech_radar
6,wikipedia,https://en.wikipedia.org/wiki/Mikoyan_MiG-29
7,wikipedia,https://en.wikipedia.org/wiki/Russian_Air_Force
8,wikipedia,https://en.wikipedia.org/wiki/Ukrainian_Air_Force
9,wikipedia,https://en.wikipedia.org/wiki/Zhuk_(radar)


In [11]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'radar'
   OR lower(object.name) CONTAINS 'radar'
   OR lower(fact.predicate) CONTAINS 'sensor'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


,subject,predicate,object,confidence,evidence
0,Bars radar family,CONTAINS_RADAR_MODEL,N035 Irbis-E,1.0,The most powerful radar of Bars radar family
1,Bars radar MAP MODE,HAS_RESOLUTION_LIMITATION,10 meters,1.0,with a maximum resolution of 10 meters
2,Bars radar,HAS_AIR_TO_AIR_DETECTION_RANGE_EARLY_WARNING_ROLE,over 400 km,1.0,The maximum air-to-air detection range is over...
3,Bars radar,HAS_AIR_TO_SURFACE_MODES_COUNT,five air-to-ground modes and four maritime modes,1.0,"For air-to-surface mode, the N011 has five air..."
4,N011M,IS_A,Bars radar family member,1.0,second member of the Bars radar family is the ...
5,Bars radar,HAS_MODE,"mapping mode using real beam, doppler beam sha...",1.0,Bars also features a mapping mode using either...
6,N011M,IS_VARIANT_OF,Bars radar family,1.0,The second member of the Bars radar family is ...
7,Bars radar (N011),HAS_MAX_TRACK_TARGETS_SUBSEQUENT_UPGRADE,15,1.0,This was subsequently upgraded to tracking 15 ...
8,Bars radar (N011),HAS_AIR_TO_AIR_DETECTION_RANGE_INTERCEPT_ROLE_...,65 km,1.0,and 65 km tail-on
9,Rezonans-NE radar,LOCATED_IN_LOCATION,Shoyna (Arctic),1.0,"constructed in the Arctic in Zapolyarniy, Indi..."


,subject,predicate,object,confidence,evidence
0,Bars radar family,CONTAINS_RADAR_MODEL,N035 Irbis-E,1.0,The most powerful radar of Bars radar family
1,Bars radar MAP MODE,HAS_RESOLUTION_LIMITATION,10 meters,1.0,with a maximum resolution of 10 meters
2,Bars radar,HAS_AIR_TO_AIR_DETECTION_RANGE_EARLY_WARNING_ROLE,over 400 km,1.0,The maximum air-to-air detection range is over...
3,Bars radar,HAS_AIR_TO_SURFACE_MODES_COUNT,five air-to-ground modes and four maritime modes,1.0,"For air-to-surface mode, the N011 has five air..."
4,N011M,IS_A,Bars radar family member,1.0,second member of the Bars radar family is the ...
5,Bars radar,HAS_MODE,"mapping mode using real beam, doppler beam sha...",1.0,Bars also features a mapping mode using either...
6,N011M,IS_VARIANT_OF,Bars radar family,1.0,The second member of the Bars radar family is ...
7,Bars radar (N011),HAS_MAX_TRACK_TARGETS_SUBSEQUENT_UPGRADE,15,1.0,This was subsequently upgraded to tracking 15 ...
8,Bars radar (N011),HAS_AIR_TO_AIR_DETECTION_RANGE_INTERCEPT_ROLE_...,65 km,1.0,and 65 km tail-on
9,Rezonans-NE radar,LOCATED_IN_LOCATION,Shoyna (Arctic),1.0,"constructed in the Arctic in Zapolyarniy, Indi..."


In [12]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'mig-29'
   OR lower(object.name) CONTAINS 'mig-29'
   OR lower(subject.name) CONTAINS 'air force'
   OR lower(object.name) CONTAINS 'air force'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


,subject,predicate,object,confidence,evidence
0,MiG-29K,HAS_MISSION,navalised operations,1.0,the navalised Mikoyan MiG-29K
1,MiG-29/35 variants,COUNT_IN_SERVICE_YEARS_2026,728,1.0,As of 2026 an estimated 728 MiG-29/35 variants...
2,Mikoyan-MiG,OPERATED_BY,Soviet Air Forces,1.0,"ration of the Soviet Union, most of the MiG-29..."
3,Russian Naval Aviation,HAS_AIRCRAFT_VARIANT,MiG-29K,1.0,A Russian Naval Aviation MiG-29K... Main artic...
4,MiG-29,OPERATED_BY_COUNTY,more than 30 nations,1.0,more than 30 nations either operate or have op...
5,Indian MiG-29 fleet maintenance issue,BASED_AT,Russia,1.0,India sent the first six of its aircraft to Ru...
6,Ukrainian Air Force,OPERATED_BY,Ukraine,1.0,Article title implies ownership by the state
7,Mikoyan MiG-29,ALSO_KNOWN_AS,Fulcrum,1.0,"""NATO reporting name : Fulcurm"""
8,Mikoyan MiG-29,HAS_CAPABILITY,multirole fighter,1.0,many MiG-29s have been furnished as multirole ...
9,MiG-29M,HAS_WEAPON_COMPATIBILITY_WITH,MiG-29K,1.0,It has a full range of weapons compatible with...


,subject,predicate,object,confidence,evidence
0,MiG-29K,HAS_MISSION,navalised operations,1.0,the navalised Mikoyan MiG-29K
1,MiG-29/35 variants,COUNT_IN_SERVICE_YEARS_2026,728,1.0,As of 2026 an estimated 728 MiG-29/35 variants...
2,Mikoyan-MiG,OPERATED_BY,Soviet Air Forces,1.0,"ration of the Soviet Union, most of the MiG-29..."
3,Russian Naval Aviation,HAS_AIRCRAFT_VARIANT,MiG-29K,1.0,A Russian Naval Aviation MiG-29K... Main artic...
4,MiG-29,OPERATED_BY_COUNTY,more than 30 nations,1.0,more than 30 nations either operate or have op...
5,Indian MiG-29 fleet maintenance issue,BASED_AT,Russia,1.0,India sent the first six of its aircraft to Ru...
6,Ukrainian Air Force,OPERATED_BY,Ukraine,1.0,Article title implies ownership by the state
7,Mikoyan MiG-29,ALSO_KNOWN_AS,Fulcrum,1.0,"""NATO reporting name : Fulcurm"""
8,Mikoyan MiG-29,HAS_CAPABILITY,multirole fighter,1.0,many MiG-29s have been furnished as multirole ...
9,MiG-29M,HAS_WEAPON_COMPATIBILITY_WITH,MiG-29K,1.0,It has a full range of weapons compatible with...


## 7. Equivalent CLI command

The same module is also exposed by the repository CLI. This notebook form is useful for iterative review, while the command below is better for repeatable batch runs.


In [ ]:
cli = [
    'python -m combat_id_calibration ingest-graph',
    f'  --neo4j-uri {NEO4J_URI}',
    f'  --neo4j-user {NEO4J_USER}',
    '  --neo4j-password "$NEO4J_PASSWORD"',
    f'  --facts-jsonl {FACTS_JSONL}',
    f'  --model {MODEL}',
    f'  --ollama-url {OLLAMA_URL}',
    *[f'  --wikipedia {url}' for url in WIKIPEDIA_URLS],
]
print(' \
'.join(cli))
